# Gen 1 / OS 2 live acceptance

Run one cell at a time; **do not use Run All**. Stop at the first error and do not retry a failed load.

## 1. Select the board and local image

Set the board address and confirm that this environment imports PyRPL and the preserved fork bitstream from this repository. This cell does not contact the board.

In [ ]:
import json
from pathlib import Path
import sys

import pyrpl
from pyrpl.redpitaya import RedPitaya

# HOSTNAME = "192.168.50.155"
HOSTNAME = r"rp-f0fcd9.local"
SSH_USER = r"root"

repo = Path(pyrpl.__file__).resolve().parents[1]
bitstream = repo / "pyrpl" / "fpga" / "red_pitaya.bin"

print("Repo:", repo)
print("Python:", sys.executable)
print("PyRPL:", pyrpl.__file__)
print("Bitstream:", bitstream)
assert bitstream.is_file()

## 2. Run the read-only preflight

Enter the SSH password. This reads the OS, `overlay.sh` contract, hardware profile, FPGA Manager state, and local asset hashes; it does not upload or program anything. Continue only if it finishes with `loader: overlay`.

In [ ]:
from getpass import getpass

ssh_password = getpass("SSH password: ")
if not ssh_password:
    raise ValueError("SSH password cannot be empty.")

preflight_device = None
try:
    preflight_device = RedPitaya(
        config=None,
        hostname=HOSTNAME,
        user=SSH_USER,
        password=ssh_password,
        gui=False,
        autostart=False,
        reloadfpga=False,
        reloadserver=False,
    )
    preflight_report = preflight_device.preflight_fpga_update()
finally:
    if preflight_device is not None:
        preflight_device.end_ssh()

assert preflight_report["read_only"] is True
assert preflight_report["loader"] == "overlay", preflight_report
print(json.dumps(preflight_report, indent=2, sort_keys=True))

## 3. Confirm physical readiness

Use recoverable OS media and disconnect outputs/actuators first. Change the confirmation value only when ready.

In [ ]:
CONFIRM_FPGA_LOAD = ""
if CONFIRM_FPGA_LOAD != "LOAD":
    raise RuntimeError("Not ready: set CONFIRM_FPGA_LOAD to 'LOAD'.")
print("Physical preparation confirmed.")

## 4. Program the FPGA and connect

This uploads the fork BIN and matched DTBO, temporarily stops the web services, programs the FPGA, starts the PyRPL server, and checks the fork register map.

In [ ]:
from pyrpl import Pyrpl

if globals().get("CONFIRM_FPGA_LOAD") != "LOAD":
    raise RuntimeError("Run the confirmation cell first.")

p = Pyrpl(
    config="gen1-os2-field-test",
    hostname=HOSTNAME,
    user=SSH_USER,
    password=ssh_password,
    filename=str(bitstream),
    loglevel="info",
    gui=False,
    reloadfpga=True,
    reloadserver=True,
)

rp = p.rp
print("FPGA load, monitor-server start, and fork register checks completed.")

## 5. Check the result

Success means the load cell returns normally, logs successful overlay and Red Pitaya connections, and prints its final message. The web UI should return after its temporary stop, but the functional checks below—not the web UI alone—prove PyRPL operation.

## 6. Optionally open the PyRPL GUI

Run this only if you want the desktop GUI; it uses the connection created above.

In [2]:
%gui qt
p.show_gui()

## 7. Read one module setting

This quick, non-output-driving check reads PID0's input-filter configuration through the live PyRPL connection.

In [2]:
pid = p.rp.pid0
print(pid.inputfilter)

[0, 0, 0]


## Test 1: DC gain and offset calibration

Loop OUT1 to IN1 and measure OUT1 with the Rigol. PID0 sweeps five DC commands while PID1 reads IN1. The saved result for this setup was `V_rigol = 1.14142 * V_rp_output + 0.001534`; treat it as board- and setup-specific.

In [8]:
import time
import numpy as np

# PID0: DC output source
src = rp.pid0
src.input  = "in1"
src.output_direct = "out1"
src.p = 0
src.i = 0
src.setpoint = 0
src.ival = 0
src.min_voltage = -0.99
src.max_voltage = 0.99
src.paused = False

#PID1: reads IN1 through PID path, outputs nowhere
mon = rp.pid1
mon.input = "in1"
mon.output_direct = "off"
mon.p = 1
mon.i = 0
mon.setpoint = 0
mon.ival = 0
mon.min_voltage = -0.99
mon.max_voltage = 0.99
mon.inputfilter = [0,0,0,0]
mon.paused = False

### Optional single-point output

This sets the PID0 DC command to +0.5 RP units. Skip it if you only need the sweep.

In [16]:
src.ival = 0.5

### Run the five-point sweep

This steps both polarities and prints the OUT1 command plus two IN1 readbacks for comparison with the Rigol.

In [9]:
for v in [-0.5, -0.25, 0.0, 0.25, 0.5]:
    src.ival = v
    time.sleep(0.5)

    scope_vals =[]
    pidmon_vals = []
    for _ in range(200):
        scope_vals.append(rp.scope.voltage_in1)
        pidmon_vals.append(mon.current_output_signal)
        time.sleep(0.005)
    print(
        "src_ival = ", v,
        "src_out = ", round(src.current_output_signal,4),
        "scope_in1_mean = ", round(float(np.mean(scope_vals)),4),
        "pid1_monitor_mean = ",  round(float(np.mean(pidmon_vals)),4),
        "pid1_std = ", round(float(np.std(pidmon_vals)),5),)

src_ival =  -0.5 src_out =  -0.5001 scope_in1_mean =  -0.5021 pid1_monitor_mean =  -0.5022 pid1_std =  0.00072
src_ival =  -0.25 src_out =  -0.25 scope_in1_mean =  -0.2516 pid1_monitor_mean =  -0.2518 pid1_std =  0.00103
src_ival =  0.0 src_out =  0.0 scope_in1_mean =  -0.003 pid1_monitor_mean =  -0.003 pid1_std =  0.00073
src_ival =  0.25 src_out =  0.25 scope_in1_mean =  0.2466 pid1_monitor_mean =  0.2467 pid1_std =  0.00072
src_ival =  0.5 src_out =  0.5001 scope_in1_mean =  0.4964 pid1_monitor_mean =  0.4964 pid1_std =  0.00072


## Test 2: dynamic triangle-wave verification

Disconnect PID0, generate a zero-centered 1 kHz triangle on OUT1 with ASG0, and sample the IN1 loopback. Use the Rigol only as a dynamic sanity check; Python polling aliases this waveform and is not a precision calibration.

In [17]:
rp.pid0.output_direct = "off"

### Start the triangle output

This routes a 1 kHz, 0.4-unit triangle from ASG0 to OUT1.

In [27]:
asg = rp.asg0

asg.output_direct = "out1"
asg.waveform = "ramp"
asg.frequency = 1000 # 1 KHz
asg.amplitude = 0.4
asg.offset = 0.0
asg.trigger_source = "immediately"

### Read the loopback

This samples IN1 and prints its observed mean, limits, and peak-to-peak span.

In [28]:
vals = []
for _ in range(5000):
    vals.append(rp.scope.voltage_in1)
    time.sleep(0.0005)
vals = np.array(vals)

print("RP mean = ", np.mean(vals))
print("RP min = ", np.min(vals))
print("RP max = ", np.max(vals))
print("RP Vpp = ", np.max(vals)-np.min(vals))

RP mean =  -0.0475296630859375
RP min =  -0.4014892578125
RP max =  0.3994140625
RP Vpp =  0.8009033203125


## Test 3: digital-setpoint PID

Load the saved calibration helpers, configure PID0 to drive OUT1 from IN1, clamp its integrator, and monitor the loop. Verify feedback polarity and safe output limits first.

In [2]:
# Test 1 output calibration: RP OUT1 command/readback -> Rigol volts.
RP_OUT_TO_RIGOL_GAIN = 1.14142
RP_OUT_TO_RIGOL_OFFSET = 0.001534

def rp_to_rigol(v_rp):
    return RP_OUT_TO_RIGOL_GAIN * v_rp + RP_OUT_TO_RIGOL_OFFSET

def rigol_to_rp(v_rigol):
    return (v_rigol - RP_OUT_TO_RIGOL_OFFSET) / RP_OUT_TO_RIGOL_GAIN

# IN1 ADC/readback calibration: use for IN1 displays and digital PID setpoints.
RP_IN1_TO_RIGOL_GAIN = 1.14380
RP_IN1_TO_RIGOL_OFFSET = 0.004668

def rp_in1_to_rigol(v_rp_in1):
    return RP_IN1_TO_RIGOL_GAIN * v_rp_in1 + RP_IN1_TO_RIGOL_OFFSET

def rigol_to_rp_in1(v_rigol):
    return (v_rigol - RP_IN1_TO_RIGOL_OFFSET) / RP_IN1_TO_RIGOL_GAIN

### Start the controller

This configures PID0 for the calibrated 0.5 V setpoint and enables OUT1.

In [15]:
pid = p.rp.pid0

pid.input = "in1"
pid.output_direct = "out1"
#pid.inputfilter = [0,0,0,0]
pid.min_voltage, pid.max_voltage = -1, 1
pid.pause_gains = "pi"
pid.paused = False

#K=1
pid.p = -0.3
pid.i = -180000
pid.ival = 0

desired_scope_voltage = 0.5
pid.setpoint = rigol_to_rp_in1(desired_scope_voltage)

print("raw RP setpoint = ", pid.setpoint)
print("scope-corrected setpoint = ", rp_in1_to_rigol(pid.setpoint))
print("execute!")

raw RP setpoint =  0.43310546875
scope-corrected setpoint =  0.5000540351562499
execute!


### Clamp the integrator

Run this once to bring the current integral value inside the configured safe range.

In [ ]:
IVAL_MIN = -1
IVAL_MAX = 1
if pid.ival< IVAL_MIN:
    pid.ival = IVAL_MIN
elif pid.ival > IVAL_MAX:
    pid.ival = IVAL_MAX

### Monitor the loop

This prints the calibrated input, setpoint, output, and integral state for about 90 seconds.

In [ ]:
import time
for _ in range(900):
    raw_in = rp.scope.voltage_in1
    corrected_in = rp_in1_to_rigol(raw_in)
    corrected_set = rp_in1_to_rigol(pid.setpoint)

    print(
        "raw in = ", round(raw_in,4),
        "scope in = ", round(corrected_in,4),
        "scope_set = ", round(corrected_set,4),
        "out = ", round(pid.current_output_signal,4),
        "ival = ", round(pid.ival,4)
    )
    time.sleep(0.1)

## Analog setpoint PID mode

Connect the process signal to IN1 and the analog reference to IN2. PID1 drives OUT1 from their FPGA-side difference; Python calibration helpers are not in that real-time path. Check polarity, limits, wiring, and actuator safety before running.

In [3]:
# Reference path: IN2 -> PID0 filtered input; no direct output.
analog_reference = rp.pid0
analog_pid = rp.pid1
analog_reference.input = "in2"
analog_reference.inputfilter = []  # Disable all stages; match both differential paths.
analog_reference.output_direct = "off"
analog_reference.p = 0
analog_reference.i = 0
analog_reference.ival = 0
analog_reference.paused = False

# Controller path: error = IN1 - IN2, 
analog_pid.input = "in1"
analog_pid.inputfilter = []  # Must match the reference path.
analog_pid.differential_mode_enabled = True
analog_pid.output_direct = "out1"
analog_pid.min_voltage = -1
analog_pid.max_voltage = 1
analog_pid.p = -0  # Verify the sign for the physical plant.
analog_pid.i = -400000
analog_pid.ival = 0

rp.hk.expansion_P1_output = False  # Configure DIO1_P as input
analog_pid.pause_gains = "pi"      # Freeze both P and I
analog_pid.paused = False     
#analog_pid.pause_gains = "off"  # Lock/hold disabled.

print("Continuous analog-setpoint PID enabled; lock/hold is disabled.")

Continuous analog-setpoint PID enabled; lock/hold is disabled.


## Stop all direct outputs

Run this immediately after a test or an exception to disconnect every available generator/controller from the physical outputs.

In [14]:
for name in ["asg0", "asg1", "pid0", "pid1","pid2", "iq0", "iq1", "iq2", "iir"]:
 try:
     m = getattr(rp, name)
     m.output_direct = "off"
     print(name, "-> off")
 except Exception as e:
    print(name, "skip: ", e)

asg0 -> off
asg1 -> off
pid0 -> off
pid1 -> off
pid2 -> off
iq0 -> off
iq1 -> off
iq2 -> off
iir skip:  'RedPitaya' object has no attribute 'iir'


## Optional calibrated +1 V output

This intentionally drives OUT1 using the saved calibration. Connect only a known-safe load, then run **Finish safely** immediately afterward.

In [ ]:
# Drive OUT1 at +1.000 V DC using the measured OUT1 calibration above.
target_out1_voltage = 1.0
rp_dc_offset = rigol_to_rp(target_out1_voltage)

if not -1.0 <= rp_dc_offset <= 1.0:
    raise ValueError(
        f"Calibrated OUT1 command {rp_dc_offset:.6f} V is outside the ASG range."
    )

# OUT1 sums routed FPGA modules, so disconnect every other source first.
for name in ["asg1", "pid0", "pid1", "pid2", "iq0", "iq1", "iq2"]:
    getattr(rp, name).output_direct = "off"

dc_out = rp.asg0
dc_out.setup(
    waveform="dc",
    amplitude=0.0,
    offset=rp_dc_offset,
    trigger_source="immediately",
    output_direct="out1",
)

print(
    f"OUT1 set to {target_out1_voltage:.3f} V DC "
    f"(calibrated RP command: {rp_dc_offset:.6f} V)."
)

## Finish safely

Run the next cell after every output-driving test, including after an exception. Verify the physical outputs are safe before disconnecting or starting another test.

In [ ]:
for name in ["asg0", "asg1", "pid0", "pid1", "pid2", "iq0", "iq1", "iq2"]:
    try:
        getattr(rp, name).output_direct = "off"
        print(name, "-> off")
    except Exception as error:
        print(name, "could not be disabled:", error)

## Reconnect without reloading

After the load and functional checks have succeeded, first run **Finish safely**, restart the notebook kernel, and run the next cell. It reconnects to the already-running PyRPL server; it does not upload or program the FPGA.

In [ ]:
from getpass import getpass
from pyrpl import Pyrpl

HOSTNAME = "192.168.50.155"
SSH_USER = "root"
SSH_PASSWORD = getpass("SSH password: ")

p = Pyrpl(
    config="gen1-os2-field-test",
    hostname=HOSTNAME,
    user=SSH_USER,
    password=SSH_PASSWORD,
    gui=False,
    reloadfpga=False,
    reloadserver=False,
)
rp = p.rp
print("Reconnected without reloading the FPGA or server.")